# Simulator backtest

Does the simulator's 10th to 90th percentile band hold the real outcome about 80% of the time?

For every real SIP window of 3, 5 and 10 years in the Nifty 50 history (overlapping, one per start month), we work out what ₹1 a month really became, divided by the money put in, and check whether it falls inside the simulated band.

Caveats: windows overlap heavily, so they are not independent. There are only a handful of **non-overlapping** windows, and just one for 10 years, so the 10-year check is weak. The bootstrap draws from the same history, so this is an in-sample check.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from core.simulator import backtest, load_monthly_returns

data = load_monthly_returns(ROOT / "data" / "nifty_monthly.csv")
returns = data["return"].to_numpy()
print(f"{len(data)} monthly returns, {data['month'].iloc[0]} to {data['month'].iloc[-1]}")

In [ ]:
results = [backtest(returns, years) for years in (3, 5, 10)]
pd.DataFrame(results).set_index("years")

In [ ]:
out = {
    "data": {"first_month": data["month"].iloc[0], "last_month": data["month"].iloc[-1], "months": len(data)},
    "target_coverage": 0.8,
    "windows": results,
}
text = json.dumps(out, indent=2)
(ROOT / "models" / "simulator_backtest.json").write_text(text + chr(10), encoding="utf-8")
print(text)